
# Notebook 19: Weather Intelligence

## Objective
Develop a reusable weather intelligence component for the BharatCrop web application using the existing integrated APY and historical weather dataset.

## Key Goals
1. Validate weather data coverage and quality.
2. Analyze historical rainfall and temperature patterns across districts and seasons.
3. Investigate relationships between weather conditions and crop yield anomalies.
4. Identify crop-season combinations that may be vulnerable to unusual weather.
5. Generate interpretable summaries and reusable outputs for web application integration.

## Dataset
`data/processed/weather_regional/apy_weather_integrated_exploratory.csv`

The dataset combines agricultural production and yield records with annual district-level weather summaries.

## Methodology
- Validate data quality and temporal coverage.
- Analyze weather distributions and anomalies.
- Calculate historical yield deviations using appropriate baselines.
- Examine weather–yield associations without assuming causation.
- Evaluate predictive value only where the data supports reliable validation.
- Save reusable outputs for integration into BharatCrop.

## Expected Outputs
- Weather coverage and quality report.
- District-level historical weather summaries.
- Crop-season weather–yield relationship summaries.
- Weather-associated yield anomaly indicators.
- Exportable CSV/JSON artifacts for the web application.

## Limitations
- Annual weather summaries may not represent conditions during specific crop growth stages.
- Historical associations do not establish causation.
- This notebook does not independently forecast future weather.
- Outputs must be validated before being presented as actionable farmer advice.

## Success Criteria
The notebook produces reliable, interpretable weather intelligence from existing data and saves outputs that can be consumed by the BharatCrop web application.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence"
)

DATA_PATH = (
    PROJECT_ROOT
    / "data/processed/weather_regional/apy_weather_integrated_exploratory.csv"
)

df = pd.read_csv(DATA_PATH, low_memory=False)

print("Dataset shape:", df.shape)

required_cols = [
    "State", "District", "Crop", "Season",
    "Year", "start_year", "Yield",
    "tmin_mean_c", "tmax_mean_c",
    "precipitation_total_mm"
]

print("\nRequired columns:")
for col in required_cols:
    print(f"{col}: {'OK' if col in df.columns else 'MISSING'}")

weather_cols = [
    "tmin_mean_c", "tmin_min_c",
    "tmax_mean_c", "tmax_max_c",
    "precipitation_total_mm",
    "precipitation_mean_mm_day",
    "precipitation_max_mm_day"
]

available_weather = [c for c in weather_cols if c in df.columns]

print("\nYear range:", df["start_year"].min(), "-", df["start_year"].max())

print("\nWeather data coverage (% non-missing):")
display(
    (df[available_weather].notna().mean() * 100)
    .round(2)
    .rename("Coverage (%)")
    .to_frame()
)

print("\nSample records:")
display(df.head())


Dataset shape: (376272, 39)

Required columns:
State: OK
District: OK
Crop: OK
Season: OK
Year: OK
start_year: OK
Yield: OK
tmin_mean_c: OK
tmax_mean_c: OK
precipitation_total_mm: OK

Year range: 2013 - 2024

Weather data coverage (% non-missing):


,Coverage (%)
tmin_mean_c,85.08
tmin_min_c,85.08
tmax_mean_c,85.08
tmax_max_c,85.08
precipitation_total_mm,85.08
precipitation_mean_mm_day,85.08
precipitation_max_mm_day,85.08



Sample records:


,State,District,Crop,Season,Crop_Category,Year,Area,Production,Yield,QC_AreaZero-ProductionPositive,...,QC_Extreme_Daily_Precipitation,QC_Long_District_Grid_Distance,weather_proxy_available,_merge,apy_measurements_complete,weather_and_apy_complete,weather_alignment_method,weather_full_season_alignment_verified,weather_available,weather_match_status
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,Grouping — Verify Source Definition,2013-14,0.0,0.0,2377.0,False,...,False,False,True,both,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,Grouping — Verify Source Definition,2013-14,0.0,0.0,526.0,False,...,False,False,True,both,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,Grouping — Verify Source Definition,2013-14,0.0,0.0,923.0,False,...,False,False,True,both,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED
3,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,Individual Crop Candidate,2013-14,0.0,0.0,526.0,False,...,False,False,True,both,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED
4,Andaman And Nicobar Islands,Nicobars,Maize,Total,Individual Crop Candidate,2013-14,0.0,0.0,526.0,False,...,False,False,True,both,True,True,STARTING_CALENDAR_YEAR_PROXY,False,True,MATCHED


In [2]:

# Weather availability and alignment checks

print("WEATHER AVAILABILITY")
if "weather_available" in df.columns:
    display(
        df["weather_available"]
        .value_counts(dropna=False)
        .rename_axis("Weather available")
        .to_frame("Rows")
    )

print("\nWEATHER ALIGNMENT METHOD")
if "weather_alignment_method" in df.columns:
    display(
        df["weather_alignment_method"]
        .value_counts(dropna=False)
        .rename_axis("Alignment method")
        .to_frame("Rows")
    )

print("\nFULL-SEASON ALIGNMENT VERIFIED")
if "weather_full_season_alignment_verified" in df.columns:
    display(
        df["weather_full_season_alignment_verified"]
        .value_counts(dropna=False)
        .rename_axis("Alignment verified")
        .to_frame("Rows")
    )

print("\nWEATHER COVERAGE BY YEAR")
year_coverage = (
    df.groupby("start_year")
    .agg(
        total_rows=("District", "size"),
        weather_rows=("precipitation_total_mm", "count")
    )
)

year_coverage["weather_coverage_pct"] = (
    year_coverage["weather_rows"]
    / year_coverage["total_rows"] * 100
).round(2)

display(year_coverage)

print("\nDISTINCT DISTRICTS")
print(df[["State", "District"]].drop_duplicates().shape[0])


WEATHER AVAILABILITY


,Rows
Weather available,
True,320124
False,56148



WEATHER ALIGNMENT METHOD


,Rows
Alignment method,
STARTING_CALENDAR_YEAR_PROXY,376272



FULL-SEASON ALIGNMENT VERIFIED


,Rows
Alignment verified,
False,376272



WEATHER COVERAGE BY YEAR


,total_rows,weather_rows,weather_coverage_pct
start_year,,,
2013,31356,26677,85.08
2014,31356,26677,85.08
2015,31356,26677,85.08
2016,31356,26677,85.08
2017,31356,26677,85.08
2018,31356,26677,85.08
2019,31356,26677,85.08
2020,31356,26677,85.08
2021,31356,26677,85.08



DISTINCT DISTRICTS
751


In [3]:

# Historical weather profiles for BharatCrop-Intelligence

weather_cols = [
    "tmin_mean_c",
    "tmin_min_c",
    "tmax_mean_c",
    "tmax_max_c",
    "precipitation_total_mm",
    "precipitation_mean_mm_day",
    "precipitation_max_mm_day"
]

weather_cols = [c for c in weather_cols if c in df.columns]

# One record per district and year avoids counting each crop as a separate
# weather observation.
district_year = (
    df.groupby(["State", "District", "start_year"], as_index=False)
    .agg(
        **{col: (col, "first") for col in weather_cols},
        weather_available=("weather_available", "max")
    )
)

# Historical district-level profiles
district_weather_summary = (
    district_year.groupby(["State", "District"])
    .agg(
        years_recorded=("start_year", "nunique"),
        years_with_weather=("weather_available", "sum"),
        **{
            f"{col}_mean": (col, "mean")
            for col in weather_cols
        },
        **{
            f"{col}_std": (col, "std")
            for col in weather_cols
        }
    )
    .reset_index()
)

district_weather_summary["weather_coverage_pct"] = (
    district_weather_summary["years_with_weather"]
    / district_weather_summary["years_recorded"] * 100
).round(2)

print("District-year records:", len(district_year))
print("District profiles:", len(district_weather_summary))

print("\nSample district profiles:")
display(district_weather_summary.head())

print("\nDistricts with lowest weather coverage:")
display(
    district_weather_summary[
        ["State", "District", "years_recorded",
         "years_with_weather", "weather_coverage_pct"]
    ]
    .sort_values("weather_coverage_pct")
    .head(10)
)


District-year records: 9012
District profiles: 751

Sample district profiles:


,State,District,years_recorded,years_with_weather,tmin_mean_c_mean,tmin_min_c_mean,tmax_mean_c_mean,tmax_max_c_mean,precipitation_total_mm_mean,precipitation_mean_mm_day_mean,precipitation_max_mm_day_mean,tmin_mean_c_std,tmin_min_c_std,tmax_mean_c_std,tmax_max_c_std,precipitation_total_mm_std,precipitation_mean_mm_day_std,precipitation_max_mm_day_std,weather_coverage_pct
0,Andaman And Nicobar Islands,Nicobars,12,12,27.734625,25.930000,28.611812,30.729167,1972.460000,5.400282,45.189167,0.246107,0.300787,0.281010,0.839301,331.063903,0.906669,13.750717,100.0
1,Andaman And Nicobar Islands,North And Middle Andaman,12,12,27.151632,23.790833,29.054759,32.588333,2538.079167,6.949394,77.541667,0.166328,0.399715,0.250235,0.688197,396.553485,1.089393,20.107123,100.0
2,Andaman And Nicobar Islands,South Andamans,12,12,27.711223,25.509167,28.739475,31.370833,2140.125000,5.859391,87.728333,0.199807,0.431329,0.252903,0.827334,319.749761,0.876218,45.595606,100.0
3,Andhra Pradesh,Alluri Sitharama Raju,12,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,Andhra Pradesh,Anakapalli,12,12,22.175732,13.904167,30.934172,40.934167,1346.940833,3.687482,87.424167,0.246219,0.596100,0.348968,0.937040,146.203565,0.397646,16.253841,100.0



Districts with lowest weather coverage:


,State,District,years_recorded,years_with_weather,weather_coverage_pct
749,West Bengal,South 24 Parganas,12,0,0.0
748,West Bengal,Purulia,12,0,0.0
747,West Bengal,Purba Medinipur,12,0,0.0
3,Andhra Pradesh,Alluri Sitharama Raju,12,0,0.0
746,West Bengal,Purba Bardhaman,12,0,0.0
745,West Bengal,Paschim Medinipur,12,0,0.0
744,West Bengal,Paschim Bardhaman,12,0,0.0
28,Andhra Pradesh,Y.S.R. Kadapa,12,0,0.0
37,Arunachal Pradesh,Leparada,12,0,0.0
720,Uttarakhand,Dehradun,12,0,0.0


In [4]:

# District weather coverage audit

district_coverage = (
    district_year.groupby(["State", "District"])
    .agg(
        years_recorded=("start_year", "nunique"),
        years_with_weather=("weather_available", "sum")
    )
    .reset_index()
)

district_coverage["weather_coverage_pct"] = (
    district_coverage["years_with_weather"]
    / district_coverage["years_recorded"] * 100
).round(2)

district_coverage["coverage_status"] = np.select(
    [
        district_coverage["weather_coverage_pct"].eq(0),
        district_coverage["weather_coverage_pct"].lt(80),
        district_coverage["weather_coverage_pct"].ge(80)
    ],
    [
        "No weather data",
        "Partial coverage",
        "Good coverage"
    ],
    default="Check"
)

print("District coverage status:")
display(
    district_coverage["coverage_status"]
    .value_counts()
    .rename_axis("Status")
    .to_frame("Districts")
)

print("\nDistricts with no weather data:")
display(
    district_coverage[
        district_coverage["coverage_status"] == "No weather data"
    ].head(15)
)

print("\nOverall district-year weather availability:")
display(
    district_year["weather_available"]
    .value_counts(dropna=False)
    .rename_axis("Weather available")
    .to_frame("District-years")
)


District coverage status:


,Districts
Status,
Good coverage,640
No weather data,111



Districts with no weather data:


,State,District,years_recorded,years_with_weather,weather_coverage_pct,coverage_status
3,Andhra Pradesh,Alluri Sitharama Raju,12,0,0.0,No weather data
28,Andhra Pradesh,Y.S.R. Kadapa,12,0,0.0,No weather data
37,Arunachal Pradesh,Leparada,12,0,0.0,No weather data
45,Arunachal Pradesh,Papum Pare,12,0,0.0,No weather data
61,Assam,Darrang,12,0,0.0,No weather data
71,Assam,Kamrup,12,0,0.0,No weather data
80,Assam,Sivasagar,12,0,0.0,No weather data
82,Assam,South Salmara Mancachar,12,0,0.0,No weather data
86,Assam,West Karbi Anglong,12,0,0.0,No weather data
113,Bihar,Purbi Champaran,12,0,0.0,No weather data



Overall district-year weather availability:


,District-years
Weather available,
True,7680
False,1332


In [5]:

# Historical weather reference ranges by district

weather_reference_cols = [
    "tmin_mean_c",
    "tmax_mean_c",
    "precipitation_total_mm"
]

# One weather record per district-year; exclude rows without weather data.
valid_weather = district_year.loc[
    district_year["weather_available"].eq(True)
].copy()

# Calculate historical district-specific reference values.
weather_reference = (
    valid_weather.groupby(["State", "District"])
    .agg(
        weather_years=("start_year", "nunique"),
        **{
            f"{col}_historical_median": (col, "median")
            for col in weather_reference_cols
        },
        **{
            f"{col}_historical_p10": (
                col, lambda x: x.quantile(0.10)
            )
            for col in weather_reference_cols
        },
        **{
            f"{col}_historical_p90": (
                col, lambda x: x.quantile(0.90)
            )
            for col in weather_reference_cols
        }
    )
    .reset_index()
)

weather_reference = weather_reference.merge(
    district_coverage[
        ["State", "District", "weather_coverage_pct", "coverage_status"]
    ],
    on=["State", "District"],
    how="right"
)

print("District reference profiles:", len(weather_reference))

print("\nCoverage status:")
display(
    weather_reference["coverage_status"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .to_frame("Districts")
)

print("\nSample reference profiles:")
display(weather_reference.head(10))


District reference profiles: 751

Coverage status:


,Districts
Status,
Good coverage,640
No weather data,111



Sample reference profiles:


,State,District,weather_years,tmin_mean_c_historical_median,tmax_mean_c_historical_median,precipitation_total_mm_historical_median,tmin_mean_c_historical_p10,tmax_mean_c_historical_p10,precipitation_total_mm_historical_p10,tmin_mean_c_historical_p90,tmax_mean_c_historical_p90,precipitation_total_mm_historical_p90,weather_coverage_pct,coverage_status
0,Andaman And Nicobar Islands,Nicobars,12.0,27.647877,28.529000,1996.530,27.522753,28.367079,1593.348,27.972748,28.861534,2429.799,100.0,Good coverage
1,Andaman And Nicobar Islands,North And Middle Andaman,12.0,27.151342,28.982397,2494.900,26.978274,28.832444,2215.545,27.304617,29.319170,2761.923,100.0,Good coverage
2,Andaman And Nicobar Islands,South Andamans,12.0,27.617692,28.656603,2196.305,27.545003,28.510805,1662.531,27.899376,29.014034,2478.482,100.0,Good coverage
3,Andhra Pradesh,Alluri Sitharama Raju,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,No weather data
4,Andhra Pradesh,Anakapalli,12.0,22.109460,30.967397,1357.030,21.944112,30.463395,1189.165,22.408660,31.237831,1442.303,100.0,Good coverage
5,Andhra Pradesh,Ananthapuramu,12.0,20.793110,33.241521,672.275,20.661589,32.475169,433.002,21.504321,34.293068,875.011,100.0,Good coverage
6,Andhra Pradesh,Annamayya,12.0,20.771959,32.882370,815.605,20.572721,31.677433,505.831,21.497764,33.446881,1086.900,100.0,Good coverage
7,Andhra Pradesh,Bapatla,12.0,24.410055,33.046836,1066.840,24.111088,32.302614,814.769,24.745879,33.249937,1223.224,100.0,Good coverage
8,Andhra Pradesh,Chittoor,12.0,20.066849,31.899370,970.030,19.910936,30.575820,618.894,20.611438,32.633337,1265.601,100.0,Good coverage
9,Andhra Pradesh,Dr. B.R. Ambedkar Konaseema,12.0,24.871425,31.362783,1177.785,24.637087,30.942452,1082.595,25.201205,31.668836,1378.332,100.0,Good coverage


In [6]:

# Flag historically unusual district-year weather

weather_flags = valid_weather[
    ["State", "District", "start_year"] + weather_reference_cols
].merge(
    weather_reference,
    on=["State", "District"],
    how="left"
)

def classify_range(value, lower, upper):
    if pd.isna(value) or pd.isna(lower) or pd.isna(upper):
        return "Unavailable"
    if value < lower:
        return "Below historical range"
    if value > upper:
        return "Above historical range"
    return "Within historical range"

for col in weather_reference_cols:
    weather_flags[f"{col}_status"] = weather_flags.apply(
        lambda row: classify_range(
            row[col],
            row[f"{col}_historical_p10"],
            row[f"{col}_historical_p90"]
        ),
        axis=1
    )

print("Rainfall status:")
display(
    weather_flags["precipitation_total_mm_status"]
    .value_counts()
    .rename_axis("Status")
    .to_frame("District-years")
)

print("\nMean temperature status:")
for col in ["tmin_mean_c", "tmax_mean_c"]:
    print(f"\n{col}:")
    display(
        weather_flags[f"{col}_status"]
        .value_counts()
        .rename_axis("Status")
        .to_frame("District-years")
    )

print("\nSample flagged records:")
display(
    weather_flags[
        ["State", "District", "start_year"]
        + weather_reference_cols
        + [f"{c}_status" for c in weather_reference_cols]
    ].head(10)
)


Rainfall status:


,District-years
Status,
Within historical range,5120
Above historical range,1280
Below historical range,1280



Mean temperature status:

tmin_mean_c:


,District-years
Status,
Within historical range,5120
Below historical range,1280
Above historical range,1280



tmax_mean_c:


,District-years
Status,
Within historical range,5120
Below historical range,1280
Above historical range,1280



Sample flagged records:


,State,District,start_year,tmin_mean_c,tmax_mean_c,precipitation_total_mm,tmin_mean_c_status,tmax_mean_c_status,precipitation_total_mm_status
0,Andaman And Nicobar Islands,Nicobars,2013,27.472904,28.279671,2456.73,Below historical range,Below historical range,Above historical range
1,Andaman And Nicobar Islands,Nicobars,2014,27.517479,28.430301,1972.96,Below historical range,Within historical range,Within historical range
2,Andaman And Nicobar Islands,Nicobars,2015,27.664959,28.533370,1767.49,Within historical range,Within historical range,Within historical range
3,Andaman And Nicobar Islands,Nicobars,2016,27.918333,28.810492,2187.42,Within historical range,Within historical range,Within historical range
4,Andaman And Nicobar Islands,Nicobars,2017,27.570219,28.360055,2155.98,Within historical range,Below historical range,Within historical range
5,Andaman And Nicobar Islands,Nicobars,2018,27.644137,28.465425,1807.23,Within historical range,Within historical range,Within historical range
6,Andaman And Nicobar Islands,Nicobars,2019,27.875671,28.748110,1425.22,Within historical range,Within historical range,Below historical range
7,Andaman And Nicobar Islands,Nicobars,2020,27.608333,28.558033,2028.44,Within historical range,Within historical range,Within historical range
8,Andaman And Nicobar Islands,Nicobars,2021,27.651616,28.524630,2020.10,Within historical range,Within historical range,Within historical range
9,Andaman And Nicobar Islands,Nicobars,2022,27.585068,28.461288,2527.51,Within historical range,Within historical range,Above historical range


In [7]:

# Validate flags against historical thresholds

for col in weather_reference_cols:
    lower = f"{col}_historical_p10"
    upper = f"{col}_historical_p90"
    status = f"{col}_status"

    below_count = (
        weather_flags[col].lt(weather_flags[lower])
        & weather_flags[col].notna()
    ).sum()

    above_count = (
        weather_flags[col].gt(weather_flags[upper])
        & weather_flags[col].notna()
    ).sum()

    print(
        f"{col}: below={below_count}, above={above_count}, "
        f"within={(weather_flags[status] == 'Within historical range').sum()}"
    )

# Compact district-year output for the app
app_weather_history = weather_flags[
    [
        "State", "District", "start_year",
        "weather_years", "weather_coverage_pct", "coverage_status",
        "tmin_mean_c", "tmax_mean_c", "precipitation_total_mm",
        "tmin_mean_c_status", "tmax_mean_c_status",
        "precipitation_total_mm_status"
    ]
].copy()

app_weather_history = app_weather_history.rename(columns={
    "start_year": "year",
    "tmin_mean_c": "mean_min_temperature_c",
    "tmax_mean_c": "mean_max_temperature_c",
    "precipitation_total_mm": "annual_precipitation_mm"
})

print("\nApp output shape:", app_weather_history.shape)
display(app_weather_history.head())


tmin_mean_c: below=1280, above=1280, within=5120
tmax_mean_c: below=1280, above=1280, within=5120
precipitation_total_mm: below=1280, above=1280, within=5120

App output shape: (7680, 12)


,State,District,year,weather_years,weather_coverage_pct,coverage_status,mean_min_temperature_c,mean_max_temperature_c,annual_precipitation_mm,tmin_mean_c_status,tmax_mean_c_status,precipitation_total_mm_status
0,Andaman And Nicobar Islands,Nicobars,2013,12.0,100.0,Good coverage,27.472904,28.279671,2456.73,Below historical range,Below historical range,Above historical range
1,Andaman And Nicobar Islands,Nicobars,2014,12.0,100.0,Good coverage,27.517479,28.430301,1972.96,Below historical range,Within historical range,Within historical range
2,Andaman And Nicobar Islands,Nicobars,2015,12.0,100.0,Good coverage,27.664959,28.533370,1767.49,Within historical range,Within historical range,Within historical range
3,Andaman And Nicobar Islands,Nicobars,2016,12.0,100.0,Good coverage,27.918333,28.810492,2187.42,Within historical range,Within historical range,Within historical range
4,Andaman And Nicobar Islands,Nicobars,2017,12.0,100.0,Good coverage,27.570219,28.360055,2155.98,Within historical range,Below historical range,Within historical range


In [8]:

import json

OUTPUT_DIR = PROJECT_ROOT / "models" / "weather_intelligence"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. District-year historical weather and anomaly indicators
history_path = OUTPUT_DIR / "district_weather_history.csv"
app_weather_history.to_csv(history_path, index=False)

# 2. District historical reference ranges and coverage
reference_path = OUTPUT_DIR / "district_weather_reference.csv"
weather_reference.to_csv(reference_path, index=False)

# 3. Metadata for app integration
metadata = {
    "module": "Weather Intelligence",
    "notebook": "19_weather_intelligence.ipynb",
    "source_dataset": str(DATA_PATH.relative_to(PROJECT_ROOT)),
    "period": {
        "start_year": int(district_year["start_year"].min()),
        "end_year": int(district_year["start_year"].max())
    },
    "districts": int(district_coverage.shape[0]),
    "districts_with_good_coverage": int(
        (district_coverage["coverage_status"] == "Good coverage").sum()
    ),
    "districts_without_weather_data": int(
        (district_coverage["coverage_status"] == "No weather data").sum()
    ),
    "district_year_records_with_weather": int(len(app_weather_history)),
    "limitations": [
        "Weather uses starting-calendar-year proxy alignment.",
        "Full crop-season alignment is not verified.",
        "Historical percentile flags are not forecasts or proven crop-risk indicators.",
        "Missing weather data must be shown as unavailable."
    ]
}

metadata_path = OUTPUT_DIR / "weather_intelligence_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved outputs:")
print(history_path)
print(reference_path)
print(metadata_path)

print("\nOutput sizes:")
for path in [history_path, reference_path, metadata_path]:
    print(f"{path.name}: {path.stat().st_size:,} bytes")


Saved outputs:
C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\weather_intelligence\district_weather_history.csv
C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\weather_intelligence\district_weather_reference.csv
C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\weather_intelligence\weather_intelligence_metadata.json

Output sizes:
district_weather_history.csv: 1,298,699 bytes
district_weather_reference.csv: 129,603 bytes
weather_intelligence_metadata.json: 687 bytes


In [9]:

# Notebook 19: Final validation

expected_outputs = [
    OUTPUT_DIR / "district_weather_history.csv",
    OUTPUT_DIR / "district_weather_reference.csv",
    OUTPUT_DIR / "weather_intelligence_metadata.json"
]

print("NOTEBOOK 19 — COMPLETION REPORT")
print("-" * 40)

all_exist = True

for path in expected_outputs:
    exists = path.exists()
    print(f"{path.name}: {'OK' if exists else 'MISSING'}")
    all_exist = all_exist and exists

print("\nDistricts covered:", len(district_coverage))
print("Districts with weather:", int(
    (district_coverage["weather_coverage_pct"] > 0).sum()
))
print("Districts without weather:", int(
    (district_coverage["weather_coverage_pct"] == 0).sum()
))
print("Historical district-year records:", len(app_weather_history))

print(
    "\nSTATUS:",
    "COMPLETE — outputs saved"
    if all_exist
    else "INCOMPLETE — check missing outputs"
)


NOTEBOOK 19 — COMPLETION REPORT
----------------------------------------
district_weather_history.csv: OK
district_weather_reference.csv: OK
weather_intelligence_metadata.json: OK

Districts covered: 751
Districts with weather: 640
Districts without weather: 111
Historical district-year records: 7680

STATUS: COMPLETE — outputs saved


In [10]:

# Notebook 19 — Predictive Model: Dataset Preparation

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score
)

# Keep individual-crop records with valid yield and weather.
model_df = df.copy()

model_df = model_df[
    (model_df["Crop_Category"] == "Individual Crop Candidate")
    & (model_df["Yield"].notna())
    & (model_df["Yield"] > 0)
    & (model_df["weather_available"] == True)
].copy()

model_df["Yield"] = pd.to_numeric(model_df["Yield"], errors="coerce")
model_df["start_year"] = pd.to_numeric(
    model_df["start_year"], errors="coerce"
)

model_df = model_df.dropna(
    subset=[
        "State", "District", "Crop", "Season",
        "start_year", "Yield",
        "tmin_mean_c", "tmax_mean_c",
        "precipitation_total_mm"
    ]
)

# One historical baseline per district-crop-season.
group_cols = ["State", "District", "Crop", "Season"]

model_df = model_df.sort_values(group_cols + ["start_year"])

model_df["previous_3yr_median_yield"] = (
    model_df.groupby(group_cols)["Yield"]
    .transform(
        lambda s: s.shift(1).rolling(window=3, min_periods=2).median()
    )
)

# Label low yield relative to previous observations only.
model_df = model_df.dropna(subset=["previous_3yr_median_yield"])

model_df["low_yield"] = (
    model_df["Yield"] < 0.80 * model_df["previous_3yr_median_yield"]
).astype("int8")

print("Predictive dataset shape:", model_df.shape)
print("\nTarget distribution:")
display(
    model_df["low_yield"]
    .value_counts()
    .rename(index={0: "Not low yield", 1: "Low yield"})
    .to_frame("Rows")
)

print("\nYear range:", model_df["start_year"].min(), "-", model_df["start_year"].max())
print("\nLow-yield event rate:", round(model_df["low_yield"].mean() * 100, 2), "%")

print("\nSample records:")
display(
    model_df[
        group_cols + [
            "start_year", "Yield", "previous_3yr_median_yield",
            "tmin_mean_c", "tmax_mean_c",
            "precipitation_total_mm", "low_yield"
        ]
    ].head(10)
)


Predictive dataset shape: (107333, 41)

Target distribution:


,Rows
low_yield,
Not low yield,92409
Low yield,14924



Year range: 2015 - 2024

Low-yield event rate: 13.9 %

Sample records:


,State,District,Crop,Season,start_year,Yield,previous_3yr_median_yield,tmin_mean_c,tmax_mean_c,precipitation_total_mm,low_yield
62715,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,2015,1000.0,4846.5,27.664959,28.533370,1767.49,1
250851,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,2021,1133.0,1000.0,27.651616,28.524630,2020.10,0
62716,Andaman And Nicobar Islands,Nicobars,Maize,Total,2015,1000.0,4846.5,27.664959,28.533370,1767.49,1
250852,Andaman And Nicobar Islands,Nicobars,Maize,Total,2021,1133.0,1000.0,27.651616,28.524630,2020.10,0
94073,Andaman And Nicobar Islands,Nicobars,Moong,Rabi,2016,400.0,2000.0,27.918333,28.810492,2187.42,1
219497,Andaman And Nicobar Islands,Nicobars,Moong,Rabi,2020,3090.0,2000.0,27.608333,28.558033,2028.44,0
94074,Andaman And Nicobar Islands,Nicobars,Moong,Total,2016,400.0,2000.0,27.918333,28.810492,2187.42,1
219498,Andaman And Nicobar Islands,Nicobars,Moong,Total,2020,3090.0,2000.0,27.608333,28.558033,2028.44,0
62723,Andaman And Nicobar Islands,Nicobars,Rice,Kharif,2015,2240.0,2362.5,27.664959,28.533370,1767.49,0
94079,Andaman And Nicobar Islands,Nicobars,Rice,Kharif,2016,1430.0,2348.0,27.918333,28.810492,2187.42,1


In [11]:
# Keep named crop seasons; exclude aggregate "Total" rows.
named_seasons = ["Kharif", "Rabi", "Summer", "Whole Year", "Autumn", "Winter"]

model_df = model_df[
    model_df["Season"].isin(named_seasons)
].copy()

# Check for duplicate district-crop-season-year records.
key_cols = ["State", "District", "Crop", "Season", "start_year"]

duplicate_count = model_df.duplicated(key_cols).sum()

print("Rows after excluding aggregate seasons:", len(model_df))
print("Duplicate key records:", duplicate_count)

print("\nTarget distribution:")
display(
    model_df["low_yield"]
    .value_counts()
    .rename(index={0: "Not low yield", 1: "Low yield"})
    .to_frame("Rows")
)

print("\nLow-yield event rate:",
      round(model_df["low_yield"].mean() * 100, 2), "%")

print("\nRows by year:")
display(
    model_df.groupby("start_year")["low_yield"]
    .agg(rows="size", low_yield_rate="mean")
    .assign(low_yield_rate=lambda x: (x["low_yield_rate"] * 100).round(2))
)

Rows after excluding aggregate seasons: 60001
Duplicate key records: 0

Target distribution:


,Rows
low_yield,
Not low yield,51508
Low yield,8493



Low-yield event rate: 14.15 %

Rows by year:


,rows,low_yield_rate
start_year,,
2015,5147,25.24
2016,5449,14.41
2017,5642,10.97
2018,6005,15.84
2019,6117,15.24
2020,6261,12.07
2021,6224,13.43
2022,6464,10.33
2023,6216,13.88


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# Keep only features available to the model; never include current Yield
# or previous_3yr_median_yield as predictors.
categorical_features = ["State", "District", "Crop", "Season"]
numeric_base = ["start_year"]
numeric_weather = [
    "tmin_mean_c",
    "tmax_mean_c",
    "precipitation_total_mm"
]

train_df = model_df[model_df["start_year"] <= 2022].copy()
test_df = model_df[model_df["start_year"] >= 2023].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train years:", train_df["start_year"].min(), "-", train_df["start_year"].max())
print("Test years:", test_df["start_year"].min(), "-", test_df["start_year"].max())
print("Train event rate:", round(train_df["low_yield"].mean() * 100, 2), "%")
print("Test event rate:", round(test_df["low_yield"].mean() * 100, 2), "%")

Training rows: 47309
Test rows: 12692
Train years: 2015 - 2022
Test years: 2023 - 2024
Train event rate: 14.47 %
Test event rate: 12.98 %


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report
)

def build_model(numeric_features):
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocess = ColumnTransformer([
        ("categorical", categorical_pipe, categorical_features),
        ("numeric", numeric_pipe, numeric_features)
    ])

    return Pipeline([
        ("preprocess", preprocess),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

y_train = train_df["low_yield"]
y_test = test_df["low_yield"]

base_features = numeric_base
weather_features = numeric_base + numeric_weather

baseline_model = build_model(base_features)
weather_model = build_model(weather_features)

X_train_base = train_df[categorical_features + base_features]
X_test_base = test_df[categorical_features + base_features]

X_train_weather = train_df[categorical_features + weather_features]
X_test_weather = test_df[categorical_features + weather_features]

baseline_model.fit(X_train_base, y_train)
weather_model.fit(X_train_weather, y_train)

baseline_prob = baseline_model.predict_proba(X_test_base)[:, 1]
weather_prob = weather_model.predict_proba(X_test_weather)[:, 1]

results = pd.DataFrame([
    {
        "Model": "Baseline (no weather)",
        "ROC_AUC": roc_auc_score(y_test, baseline_prob),
        "PR_AUC": average_precision_score(y_test, baseline_prob),
        "Brier_Score": brier_score_loss(y_test, baseline_prob)
    },
    {
        "Model": "Weather-informed",
        "ROC_AUC": roc_auc_score(y_test, weather_prob),
        "PR_AUC": average_precision_score(y_test, weather_prob),
        "Brier_Score": brier_score_loss(y_test, weather_prob)
    }
])

display(results.round(4))

print("\nWeather-informed model report at threshold 0.50:")
print(classification_report(
    y_test,
    (weather_prob >= 0.50).astype(int),
    target_names=["Not low yield", "Low yield"],
    zero_division=0
))

,Model,ROC_AUC,PR_AUC,Brier_Score
0,Baseline (no weather),0.5952,0.1650,0.1760
1,Weather-informed,0.5943,0.1649,0.1845



Weather-informed model report at threshold 0.50:
               precision    recall  f1-score   support

Not low yield       0.88      0.78      0.83     11045
    Low yield       0.18      0.31      0.22      1647

     accuracy                           0.72     12692
    macro avg       0.53      0.55      0.53     12692
 weighted avg       0.79      0.72      0.75     12692



In [14]:
# Notebook 19 — Weather contribution diagnostics

weather_coef = weather_model.named_steps["classifier"].coef_[0]
feature_names = weather_model.named_steps["preprocess"].get_feature_names_out()

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": weather_coef
})

weather_coef_df = coef_df[
    coef_df["Feature"].str.contains(
        "tmin_mean_c|tmax_mean_c|precipitation_total_mm",
        regex=True
    )
].copy()

weather_coef_df["Abs_Coefficient"] = weather_coef_df["Coefficient"].abs()
weather_coef_df = weather_coef_df.sort_values(
    "Abs_Coefficient", ascending=False
)

print("Weather feature coefficients:")
display(weather_coef_df)

print("\nPredicted risk distribution:")
display(
    pd.DataFrame({
        "Baseline probability": baseline_prob,
        "Weather probability": weather_prob
    }).describe().round(4)
)

print("\nBaseline vs weather model performance:")
display(results.round(4))

Weather feature coefficients:


,Feature,Coefficient,Abs_Coefficient
654,numeric__precipitation_total_mm,-0.227696,0.227696
653,numeric__tmax_mean_c,0.095114,0.095114
652,numeric__tmin_mean_c,-0.054759,0.054759



Predicted risk distribution:


,Baseline probability,Weather probability
count,12692.0000,12692.0000
mean,0.3568,0.3727
std,0.1501,0.1540
min,0.0048,0.0043
25%,0.2476,0.2607
50%,0.3563,0.3731
75%,0.4691,0.4886
max,0.7743,0.7972



Baseline vs weather model performance:


,Model,ROC_AUC,PR_AUC,Brier_Score
0,Baseline (no weather),0.5952,0.1650,0.1760
1,Weather-informed,0.5943,0.1649,0.1845


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    classification_report
)

def build_rf_model(numeric_features):
    preprocess = ColumnTransformer([
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ),
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            numeric_features
        )
    ])

    return Pipeline([
        ("preprocess", preprocess),
        ("classifier", RandomForestClassifier(
            n_estimators=250,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        ))
    ])

rf_baseline = build_rf_model(base_features)
rf_weather = build_rf_model(weather_features)

rf_baseline.fit(X_train_base, y_train)
rf_weather.fit(X_train_weather, y_train)

rf_base_prob = rf_baseline.predict_proba(X_test_base)[:, 1]
rf_weather_prob = rf_weather.predict_proba(X_test_weather)[:, 1]

rf_results = pd.DataFrame([
    {
        "Model": "Logistic baseline",
        "ROC_AUC": roc_auc_score(y_test, baseline_prob),
        "PR_AUC": average_precision_score(y_test, baseline_prob),
        "Brier_Score": brier_score_loss(y_test, baseline_prob)
    },
    {
        "Model": "Logistic + weather",
        "ROC_AUC": roc_auc_score(y_test, weather_prob),
        "PR_AUC": average_precision_score(y_test, weather_prob),
        "Brier_Score": brier_score_loss(y_test, weather_prob)
    },
    {
        "Model": "Random Forest baseline",
        "ROC_AUC": roc_auc_score(y_test, rf_base_prob),
        "PR_AUC": average_precision_score(y_test, rf_base_prob),
        "Brier_Score": brier_score_loss(y_test, rf_base_prob)
    },
    {
        "Model": "Random Forest + weather",
        "ROC_AUC": roc_auc_score(y_test, rf_weather_prob),
        "PR_AUC": average_precision_score(y_test, rf_weather_prob),
        "Brier_Score": brier_score_loss(y_test, rf_weather_prob)
    }
])

display(rf_results.round(4))

print("\nRandom Forest + weather report at threshold 0.50:")
print(classification_report(
    y_test,
    (rf_weather_prob >= 0.50).astype(int),
    target_names=["Not low yield", "Low yield"],
    zero_division=0
))

,Model,ROC_AUC,PR_AUC,Brier_Score
0,Logistic baseline,0.5952,0.1650,0.1760
1,Logistic + weather,0.5943,0.1649,0.1845
2,Random Forest baseline,0.6174,0.1803,0.2023
3,Random Forest + weather,0.6092,0.1729,0.2052



Random Forest + weather report at threshold 0.50:
               precision    recall  f1-score   support

Not low yield       0.89      0.74      0.81     11045
    Low yield       0.18      0.37      0.24      1647

     accuracy                           0.70     12692
    macro avg       0.53      0.56      0.52     12692
 weighted avg       0.80      0.70      0.74     12692



In [16]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import numpy as np
import pandas as pd

train_rate = y_train.mean()
test_rate = y_test.mean()

constant_prob = np.full(len(y_test), train_rate)

comparison = pd.DataFrame([
    {
        "Model": "Training prevalence only",
        "ROC_AUC": roc_auc_score(y_test, constant_prob),
        "PR_AUC": average_precision_score(y_test, constant_prob),
        "Brier_Score": brier_score_loss(y_test, constant_prob)
    },
    {
        "Model": "Logistic baseline",
        "ROC_AUC": roc_auc_score(y_test, baseline_prob),
        "PR_AUC": average_precision_score(y_test, baseline_prob),
        "Brier_Score": brier_score_loss(y_test, baseline_prob)
    },
    {
        "Model": "Random Forest baseline",
        "ROC_AUC": roc_auc_score(y_test, rf_base_prob),
        "PR_AUC": average_precision_score(y_test, rf_base_prob),
        "Brier_Score": brier_score_loss(y_test, rf_base_prob)
    }
])

print("Training event rate:", round(train_rate, 4))
print("Test event rate:", round(test_rate, 4))
display(comparison.round(4))

Training event rate: 0.1447
Test event rate: 0.1298


,Model,ROC_AUC,PR_AUC,Brier_Score
0,Training prevalence only,0.5000,0.1298,0.1132
1,Logistic baseline,0.5952,0.1650,0.1760
2,Random Forest baseline,0.6174,0.1803,0.2023


In [17]:
import json
from pathlib import Path
from datetime import datetime, timezone

OUTPUT_DIR = PROJECT_ROOT / "models" / "weather_intelligence"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

evaluation = {
    "model_task": "Experimental low-yield classification",
    "target_definition": (
        "Current yield below 80% of the median of the previous "
        "2-3 available positive-yield observations for the same "
        "state, district, crop, and named season."
    ),
    "train_years": [2015, 2022],
    "test_years": [2023, 2024],
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "train_event_rate": float(y_train.mean()),
    "test_event_rate": float(y_test.mean()),
    "metrics": {
        "constant_baseline": {
            "roc_auc": 0.5,
            "pr_auc": float(average_precision_score(y_test, constant_prob)),
            "brier_score": float(brier_score_loss(y_test, constant_prob))
        },
        "logistic_baseline": {
            "roc_auc": float(roc_auc_score(y_test, baseline_prob)),
            "pr_auc": float(average_precision_score(y_test, baseline_prob)),
            "brier_score": float(brier_score_loss(y_test, baseline_prob))
        },
        "logistic_with_weather": {
            "roc_auc": float(roc_auc_score(y_test, weather_prob)),
            "pr_auc": float(average_precision_score(y_test, weather_prob)),
            "brier_score": float(brier_score_loss(y_test, weather_prob))
        },
        "random_forest_baseline": {
            "roc_auc": float(roc_auc_score(y_test, rf_base_prob)),
            "pr_auc": float(average_precision_score(y_test, rf_base_prob)),
            "brier_score": float(brier_score_loss(y_test, rf_base_prob))
        },
        "random_forest_with_weather": {
            "roc_auc": float(roc_auc_score(y_test, rf_weather_prob)),
            "pr_auc": float(average_precision_score(y_test, rf_weather_prob)),
            "brier_score": float(brier_score_loss(y_test, rf_weather_prob))
        }
    },
    "weather_added_predictive_value": False,
    "deployment_status": "NOT READY",
    "limitations": [
        "Annual district weather is aligned using a starting-calendar-year proxy.",
        "Full crop-season weather alignment is unverified.",
        "Weather did not improve held-out classification metrics.",
        "Trained-model Brier scores were worse than the constant baseline.",
        "Predicted probabilities must not be presented as validated farmer risk estimates."
    ],
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat()
}

evaluation_path = OUTPUT_DIR / "low_yield_model_evaluation.json"

with open(evaluation_path, "w", encoding="utf-8") as f:
    json.dump(evaluation, f, indent=2)

print("Saved:", evaluation_path)
print("Deployment status:", evaluation["deployment_status"])
print("Weather added predictive value:", evaluation["weather_added_predictive_value"])

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\weather_intelligence\low_yield_model_evaluation.json
Deployment status: NOT READY
Weather added predictive value: False



## Final Conclusion — Weather Intelligence

### 1. Objective
This notebook developed a district-level weather intelligence module using the existing integrated APY and annual weather dataset. It also evaluated whether weather features could improve low-yield risk prediction.

### 2. Work Completed
- Aggregated annual weather records at the state, district, and year level.
- Evaluated weather data availability across 751 districts.
- Identified 640 districts with historical weather data and 111 districts without weather data.
- Generated district-level historical weather summaries and reference ranges for minimum temperature, maximum temperature, and precipitation.
- Created descriptive historical weather flags based on district-specific percentile ranges.
- Exported weather history, reference data, and metadata for potential application integration.
- Developed and evaluated an experimental low-yield classification model.
- Saved predictive evaluation results in `models/weather_intelligence/low_yield_model_evaluation.json`.

### 3. Predictive Experiment Results
The low-yield classification experiment defined a low-yield event as a yield below 80% of the median of the previous two or three available positive-yield observations for the same state, district, crop, and named season.

The models were evaluated using a time-based split:
- Training period: 2015–2022
- Testing period: 2023–2024

The best ROC-AUC obtained was 0.6174 from the Random Forest baseline without weather features. Adding weather features reduced ROC-AUC to 0.6092. The trained models also had worse Brier scores than the constant-probability baseline.

Therefore, the experiment did not establish reliable low-yield prediction or demonstrate additional predictive value from the available weather features.

### 4. Limitations
- Weather observations are aligned to crop years using a starting-calendar-year proxy.
- Full crop-season weather alignment has not been verified.
- Annual weather summaries do not support short-term weather forecasting.
- Historical percentile flags are descriptive indicators, not validated agronomic risk thresholds.
- The low-yield classifier is experimental and is not approved for farmer-facing deployment.
- Missing weather data must be reported as unavailable rather than interpreted as normal conditions.

### 5. Final Outputs
The following artifacts were generated in `models/weather_intelligence/`:

- `district_weather_history.csv`
- `district_weather_reference.csv`
- `weather_intelligence_metadata.json`
- `low_yield_model_evaluation.json`

### 6. Final Status

**Notebook Status:** COMPLETE

**Historical Weather Intelligence:** Generated

**Predictive Experiment:** Evaluated; not ready for deployment

**Weather-Based Predictive Improvement:** Not demonstrated

**Application Integration:** Historical weather outputs may be integrated with appropriate data-coverage and limitation disclosures. Predictive risk probabilities must not be exposed as validated farmer risk estimates.

### 7. Future Improvements
Future work should prioritize verified crop-season weather alignment, more temporally relevant weather observations, stronger time-based validation, and evidence that weather features improve predictive performance before deploying weather-based agricultural warnings.

Notebook 19 establishes the historical weather intelligence foundation for BharatCrop-Intelligence while documenting the limitations of the current predictive experiment.
